AI VISION LAB · 05 / STUDENT WORKBOOK

# Mission 05. 시드를 고정하고 실험 모델 만들기

이미지 데이터 증강과 재현 가능한 실험

**학습 목표**

- 같은 CNN 코드로 학습해도 결과가 달라질 수 있는 이유를 설명할 수 있다.
- 난수 시드를 설정하고, 훈련·검증 데이터 분할과 학습 조건을 통제할 수 있다.
- 회전 각도를 전달받아 CNN 모델을 새로 생성하는 함수를 작성할 수 있다.

### VS Code 실행 준비

회전 각도를 받는 모델 생성 함수와 공통 데이터 분할 함수를 준비합니다.

커널 목록에서 TensorFlow가 설치된 Python 환경을 선택하고 위에서부터 실행하세요.

In [ ]:
from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers

COURSE_DIR = Path('D:\\2604340036 - 송성민\\2학기\\반도체장비AI비전실무\\2026-AIVISION_05강')
DATA_ROOT = COURSE_DIR.parent / "wafermap_image_data_261021"

IMG_SIZE = 64
class_names = ["Center", "Edge-Loc", "Edge-Ring", "Loc", "Scratch"]
BATCH_SIZE = 32
TRAIN_DIR = DATA_ROOT / "Train"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

def check_image_folders(directory):
    missing = [
        name for name in class_names
        if not any(
            path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            for path in (directory / name).rglob("*")
        )
    ]
    if missing:
        raise FileNotFoundError(
            f"{directory}에 이미지가 없는 클래스: {', '.join(missing)}. "
            "첫 셀의 DATA_ROOT를 4강 Train/Test가 있는 폴더로 설정하세요."
        )

## 1. 같은 코드인데 왜 결과가 달라질까?

Mission 04에서는 회전 증강 CNN을 학습하고 기존 CNN과 정확도를 비교했다. 그런데 **같은 코드를 다시 실행해도 결과가 조금씩 달라질 수 있다.** CNN 학습 과정에 무작위로 결정되는 요소가 있기 때문이다.

| 무작위 요소 | 어디에서 사용될까? |
| --- | --- |
| 초기 가중치 | 새 CNN을 생성할 때 가중치의 시작값이 정해짐 |
| 데이터 셔플 | 학습 이미지의 순서를 섞음 |
| 무작위 회전 | 학습 중 이미지마다 회전 각도가 결정됨 |

**실험 A**

시드 42

초기값 · 셔플 · 회전 난수

회전 범위: ±15°

**실험 B**

시드 42

초기값 · 셔플 · 회전 난수

회전 범위: ±45°

두 실험이 가능한 한 같은 조건에서 시작하도록 하고, **회전 범위만** 바꾸면 그 차이를 해석하기 쉬워진다. 이를 **실험 조건 통제**라고 한다.

## 2. 난수 시드(Random Seed) 이해하기
컴퓨터의 난수는 일정한 알고리즘으로 생성하는 의사난수인 경우가 많다. 시드는 난수 생성을 시작할 기준값이다. Keras에서는 한 줄로 Python, NumPy, TensorFlow의 주요 난수 시드를 설정할 수 있다.

In [ ]:
SEED = 42
keras.utils.set_random_seed(SEED)

`42`는 특별한 성능을 가진 숫자가 아니다. **같은 조건으로 비교하기 위해 실험들에서 동일한 값을 사용한다**는 것이 중요하다.
시드를 고정해도 모든 실행 환경에서 결과가 완전히 같아지는 것은 아니다. GPU 연산, 라이브러리 버전, 실행 환경에 따라 차이가 날 수 있다. 이번 실습의 목적은 무작위 조건을 최대한 통제하는 방법을 배우는 것이다.
### 시드를 한 번만 설정하면 안 될까?
실험을 연속 실행하면 난수 생성 상태가 진행된다. 따라서 실험마다 새 모델을 만들기 직전 시드를 다시 설정하자.
여기서 **“실험을 연속 실행한다”**는 말은 예를 들어 **±15° 모델을 생성·학습하고, 이어서 ±45° 모델을 생성·학습하고, 다시 ±90° 모델을 생성·학습**하는 과정을 뜻한다.
`SEED = 42`에 저장된 숫자가 변하는 것은 아니다. 난수 생성기는 **42를 출발점으로 난수를 순서대로 만들어 낸다.** 모델 가중치를 초기화하거나 이미지를 무작위로 회전할 때 난수가 사용되면, 생성 상태가 다음 위치로 진행된다.
시드 설정: 42난수 생성의 시작점 지정

↓

난수 ① → 난수 ② → 난수 ③ → …사용할수록 다음 순서로 진행

↓

시드를 다시 42로 설정난수 생성의 출발점으로 다시 시작
따라서 시드를 맨 처음 한 번만 설정한 채 네 모델을 차례대로 만들면, 각 모델의 가중치 초기화에 서로 다른 난수 상태가 사용될 수 있다. **회전 범위 외의 조건을 최대한 맞추기 위해 새 모델을 만들 때마다 시드를 다시 설정한다.**

실험마다 시드를 다시 설정하는 위치를 보여주는 예시입니다.

```python
for degree in [0, 15, 45, 90]:
    keras.utils.set_random_seed(SEED)  # 매 실험 시작 시 다시 설정
    model_exp = create_cnn_model(degree)  # 새 모델 생성
    # 이후 학습 진행
```

이번 워크북의 `create_cnn_model()` 안에도 시드 설정이 들어 있으므로, 그 함수를 호출하면 모델 초기화 직전에 시드가 다시 설정된다. 위 코드는 **언제 시드를 다시 설정하는지** 보여주는 예시다.

시드를 다시 설정한다는 것은 모든 회전 이미지가 똑같아진다는 뜻이 아니다. 회전 범위가 달라지면 생성되는 각도의 범위도 달라진다. 또한 실제 학습 결과의 완전한 일치는 실행 환경에 따라 보장되지 않는다.

## 3. 훈련·검증 데이터가 겹치지 않게 나누기
4강에서 `image_dataset_from_directory()`로 훈련 데이터와 검증 데이터를 분리했던 설정을 기억해보자. 이번에는 `subset="both"`를 사용하여 두 데이터셋을 **하나의 호출**로 만든다.
### 왜 데이터셋 생성 함수를 따로 만들까?
회전 각도별로 CNN을 새로 학습하려면, 매번 **같은 훈련·검증 데이터와 동일한 셔플 설정**을 사용하는 것이 중요하다. 데이터셋 생성 코드를 매번 복사하면 시드나 분할 비율을 실수로 다르게 설정할 수 있다.
그래서 데이터셋을 만드는 코드를 `create_experiment_datasets()`라는 함수로 묶는다. 함수는 호출할 때마다 동일한 **설정**으로 훈련용·검증용 데이터셋을 다시 만들고, 두 결과를 반환한다.
create_experiment_datasets()동일한 설정으로 호출

↓

train_exp + val_exp훈련 80% · 검증 20%
단, **함수로 만들었다는 이유만으로 무작위 순서가 자동 고정되지는 않는다.** 코드에서 `seed=SEED`를 지정하고, 실험마다 같은 조건으로 생성해야 재현 가능성을 높일 수 있다.

In [ ]:
import numpy as np
import tensorflow as tf

def list_images(directory):
    check_image_folders(directory)
    image_paths = []
    labels = []
    for label, class_name in enumerate(class_names):
        class_paths = sorted(
            path for path in (directory / class_name).rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        image_paths.extend(class_paths)
        labels.extend([label] * len(class_paths))
    return image_paths, np.asarray(labels, dtype=np.int32)

def decode_image(image_bytes, label):
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label

def make_dataset(image_paths, labels, shuffle=False, seed=42):
    # Python이 파일을 읽어 Windows의 한글 경로를 처리합니다.
    image_bytes = [path.read_bytes() for path in image_paths]
    dataset = tf.data.Dataset.from_tensor_slices((image_bytes, labels))
    if shuffle:
        dataset = dataset.shuffle(BATCH_SIZE * 8, seed=seed)
    dataset = dataset.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def create_experiment_datasets():
    image_paths, labels = list_images(TRAIN_DIR)

    # 같은 파일 목록과 시드로 학습·검증을 80:20으로 나눕니다.
    indices = np.random.RandomState(SEED).permutation(len(image_paths))
    val_count = int(len(indices) * 0.2)
    train_indices, val_indices = indices[:-val_count], indices[-val_count:]

    train_exp = make_dataset(
        [image_paths[i] for i in train_indices], labels[train_indices],
        shuffle=True, seed=SEED,
    )
    val_exp = make_dataset([image_paths[i] for i in val_indices], labels[val_indices])
    return train_exp, val_exp

| 설정 | 의미 |
| --- | --- |
| `validation_split=0.2` | 훈련 폴더 이미지를 약 8:2로 나눔 |
| `subset="both"` | 훈련용과 검증용 데이터셋을 함께 반환 |
| `seed=SEED` | 파일 섞기 및 분할에 사용할 시드 지정 |
| `shuffle=True` | 이미지를 섞어 데이터셋 구성 |

기존 4강 코드처럼 `subset="training"`과 `subset="validation"`으로 각각 호출해도 되지만, 이 경우 경로·시드·셔플·분할 비율을 같은 값으로 맞춰야 한다.

**주의:** 훈련 데이터에만 `shuffle=False`를 설정하고 검증 데이터는 `shuffle=True`로 두면 두 호출의 분할 대상이 어긋날 수 있다. 또한 서로 다른 파일에 같은 이미지가 복사되어 있다면 파일 단위 분할만으로 중복 이미지는 제거되지 않는다.

## 4. 회전 각도를 받는 CNN 생성 함수 작성하기
회전 각도를 바꾸어 실험할 때마다 CNN 코드를 복사하면 실수하기 쉽다. 모델 생성 과정을 함수로 묶고, **회전 각도만 전달인자로 받도록** 만들어보자.
create_cnn_model(45)회전 각도 45를 함수에 전달

↓

RandomRotation(factor=45/360)나머지 CNN 구조는 동일하게 유지

↓

새 CNN 모델 반환아직 학습된 모델은 아님
**코드 작성 미션**: 4강 CNN 코드의 Conv2D, MaxPooling2D, Flatten, Dense 부분을 그대로 재사용한다. 아래 함수에서 증강 레이어 다음 부분을 직접 완성하자.

In [ ]:
def create_cnn_model(rotation_degree=0):
    # 모델 초기화 전에 난수 시드 고정
    keras.utils.set_random_seed(SEED)

    model = keras.Sequential([
        keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.RandomRotation(
            factor=rotation_degree / 360,
            fill_mode="constant",
            fill_value=0.0,
            interpolation="nearest"
        ),
        layers.Rescaling(1./255),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(len(class_names), activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

`rotation_degree=0`이면 회전 범위가 0이므로 이미지가 회전하지 않는다. 이 모델에도 RandomRotation 레이어가 있지만 회전 효과는 없는 상태다.

**왜 함수 안에 시드 설정을 넣었을까?** 이 함수를 호출할 때마다 모델의 초기화 조건을 가능한 한 동일하게 시작하기 위해서다. 동일한 모델 구조, 동일한 학습 조건을 사용하고 회전 범위만 바꿀 수 있다.

## 5. 모델을 생성하고 구조 확인하기
함수를 완성했다면 모델을 하나 생성해보자. 아직 `fit()`은 실행하지 않는다.

In [ ]:
model_exp = create_cnn_model(45)
model_exp.summary()

**확인할 내용**

1. `summary()`에서 `RandomRotation` 레이어를 찾는다.
2. 4강 CNN과 Conv2D, MaxPooling2D, Dense 등의 순서가 같은지 확인한다.
3. `create_cnn_model(15)`로 바꿔 호출해도 모델 구조가 같은지 확인한다.

함수에서 새 모델을 만들기 때문에 이미 학습한 `model_aug`나 4강 모델에 이어서 학습하는 것이 아니다. **각 회전 범위를 처음부터 학습시키기 위한 준비**이다.

## 6. 코드 이해 및 실험 설계 활동

**Q5.1.**

같은 CNN을 두 번 학습해도 정확도가 달라질 수 있는 이유를 위 표에서 두 가지 찾아 작성하시오.

**답변**

같은 CNN이라도 **초기 가중치**와 **학습 데이터의 셔플 순서**가 달라지면 학습 결과가 달라질 수 있습니다.

**Q5.2.**

`keras.utils.set_random_seed(SEED)`를 모델 생성 함수 안에서 실행하는 이유는 무엇인가? 각 실험 시작 시 한 번씩 실행하도록 해도 되는가?

**답변**

모델을 만들 때마다 같은 난수 상태에서 시작하도록 함수 안에서 시드를 설정합니다. 각 실험에서 **새 모델을 만들기 직전**에 같은 시드로 다시 설정해도 됩니다.

**Q5.3.**

`create_experiment_datasets()`에서 훈련 데이터와 검증 데이터를 겹치지 않게 나누는 데 관여하는 설정을 찾아 쓰시오. `subset="both"`가 반환하는 값은 몇 개인가?

**답변**

`validation_split=0.2`, `subset="both"`, `seed=SEED`로 같은 원본 파일을 훈련과 검증에 나눕니다. `subset="both"`는 **훈련 데이터셋과 검증 데이터셋 두 개**를 반환합니다.

**Q5.4.**

±15°, ±45°, ±90° 회전을 적용하려면 `create_cnn_model()`에 각각 어떤 값을 전달해야 하는가? 모델 내부에서는 그 값을 어떻게 변환하는가?

**답변**

`create_cnn_model()`에 각각 `15`, `45`, `90`을 전달합니다. 함수는 `rotation_degree / 360`으로 계산해 `RandomRotation`의 `factor`로 사용합니다.

**Q5.5.**

회전 각도의 효과를 비교할 때 회전 범위 외에 동일하게 유지해야 할 조건을 세 가지 이상 작성하시오.

**답변**

**훈련·검증 분할, 난수 시드, CNN 구조, 입력 크기, 배치 크기, 에포크 수, 옵티마이저**를 동일하게 유지합니다.

**Q5.6.**

이번 미션에서 `create_cnn_model(45)`를 실행한 직후의 모델은 학습이 끝난 상태인가? 실제로 학습하려면 어떤 메서드를 사용해야 하는가?

**답변**

모델 구조와 학습 설정을 준비한 상태입니다. `model_exp.fit()`에 훈련 데이터와 검증 데이터를 전달해 학습합니다.